# NitroGen 2D Platformer Gameplay Training with ConvNeXt-RWKV7

This notebook builds a complete end-to-end pipeline to:
1. **Filter 2D Platformer Games** from [NVIDIA NitroGen](https://huggingface.co/datasets/nvidia/NitroGen) (Mario, Celeste, Hollow Knight, Cuphead, Sonic, Mega Man, Shovel Knight, Dead Cells, etc.).
2. **Download Platformer Gameplay Videos** using the URLs in `metadata.json` via `yt-dlp`.
3. **Extract & Synchronize Video Frames** at the exact annotated timestamps into `/kaggle/working/data/nitrogen_platformer_frames`.
4. **Train the ConvNeXt-RWKV7 Gamepad Model** with DINOv3 pre-trained visual priors on Kaggle **2x T4 GPUs** using PyTorch Lightning DDP with mixed precision.
5. **Run Real-Time Online Recurrent Streaming Inference** on game frames.

### Model Architecture
- **_InputNormalize:** DINOv3 ImageNet mean/std normalization
- **AdaptiveLearnedPool2d:** Adaptive spatial downsampling to 224x224
- **ConvNeXt Backbone:** DINOv3 pre-trained representations (`facebook/dinov3-convnext-tiny-pretrain-lvd1689m`)
- **LearnedWeightedGAP:** Spatial attention pooling + global average pooling
- **CausalConv1d:** Temporal convolution with residual shortcut
- **4x RWKV-7 Blocks:** Linear attention recurrent temporal mixing
- **GamepadHead:** 21-D output (17 button logits via BCE + 4 joystick axes in [-1.0, 1.0] via MSE)


In [ ]:
# Install ConvNeXt Platform, yt-dlp for video downloads, and OpenCV for frame extraction
%pip install -q "git+https://github.com/Gabz4200/ConvNeXt_Platform.git" yt-dlp opencv-python-headless


In [ ]:
import os
import json
import tarfile
import subprocess
from pathlib import Path
from functools import partial

import cv2
import torch
import lightning as L
from huggingface_hub import HfFileSystem
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping, RichProgressBar
from lightning.pytorch.loggers import CSVLogger

# Enable safe checkpoint unpickling on PyTorch 2.6+
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")

# Authenticate with Hugging Face Hub using Kaggle secret 'HF_TOKEN' for DINOv3 access
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    from huggingface_hub import login
    login(token=hf_token)
    print("Successfully authenticated with Hugging Face Hub via Kaggle secret (HF_TOKEN)!")
except Exception as e:
    hf_token = os.environ.get("HF_TOKEN")
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print("Authenticated with Hugging Face Hub via environment variable HF_TOKEN.")
    else:
        print(f"Hugging Face Hub authentication notice: {e}")

L.seed_everything(3407, workers=True)

num_gpus = torch.cuda.device_count()
print(f"PyTorch Version: {torch.__version__}")
print(f"Lightning Version: {L.__version__}")
print(f"Available GPUs: {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")


In [ ]:
# Scan NitroGen shards from Hugging Face and filter for 2D Platformer games
PLATFORMER_KEYWORDS = [
    "mario", "celeste", "hollow knight", "cuphead", "sonic",
    "shovel knight", "rayman", "megaman", "mega man", "dead cells",
    "ori", "castlevania", "metroid", "spelunky", "super meat boy",
    "donkey kong", "crash bandicoot", "kirby", "guacamelee", "inside",
    "limbo", "fez", "broforce", "terraria", "jump king", "pizza tower",
    "platformer", "2d platformer",
]

def is_platformer(game_name: str) -> bool:
    name_lower = game_name.lower()
    return any(kw in name_lower for kw in PLATFORMER_KEYWORDS)

fs = HfFileSystem()
max_shards_to_scan = 3  # Increase to 10-100 on full Kaggle runs
matched_chunks = []

print(f"Scanning first {max_shards_to_scan} NitroGen shards for platformer games...")
for shard_idx in range(max_shards_to_scan):
    shard_path = f"datasets/nvidia/NitroGen/actions/SHARD_{shard_idx:04d}.tar.gz"
    try:
        with fs.open(shard_path, "rb") as f, tarfile.open(fileobj=f, mode="r|gz") as tar:
            for member in tar:
                if member.name.endswith("metadata.json"):
                    extracted = tar.extractfile(member)
                    if extracted:
                        meta = json.load(extracted)
                        game = meta.get("game", "")
                        orig = meta.get("original_video")
                        if is_platformer(game) and orig:
                            matched_chunks.append({
                                "shard_idx": shard_idx,
                                "game": game,
                                "uuid": meta.get("uuid"),
                                "chunk_id": meta.get("chunk_id"),
                                "video_id": orig.get("video_id"),
                                "url": orig.get("url"),
                                "start_time": orig.get("start_time", 0.0),
                                "end_time": orig.get("end_time", 20.0),
                                "start_frame": orig.get("start_frame", 0),
                                "end_frame": orig.get("end_frame", 1200),
                            })
    except (tarfile.TarError, OSError) as err:
        logger.warning("Error reading shard %d: %s", shard_idx, err)

print(f"Found {len(matched_chunks)} platformer chunks across scanned shards!")
for c in matched_chunks[:5]:
    print(f"  Game: {c['game']} | Video ID: {c['video_id']} | URL: {c['url']}")


In [ ]:
# Download gameplay videos and extract synchronized frames
video_frames_dir = Path("/kaggle/working/data/nitrogen_platformer_frames")
video_frames_dir.mkdir(parents=True, exist_ok=True)
downloaded_count = 0
max_videos_to_download = 10  # Set to desired number of platformer videos

for chunk in matched_chunks:
    if downloaded_count >= max_videos_to_download:
        break

    vid_id = chunk["video_id"]
    url = chunk["url"]
    start_frame = chunk["start_frame"]
    end_frame = chunk["end_frame"]
    start_time = chunk["start_time"]
    end_time = chunk["end_time"]

    if not url or not vid_id:
        continue

    frame_dest_dir = video_frames_dir / vid_id
    if (frame_dest_dir / f"frame_{start_frame:06d}.jpg").exists():
        print(f"Frames already extracted for {vid_id}, skipping download.")
        downloaded_count += 1
        continue

    frame_dest_dir.mkdir(parents=True, exist_ok=True)
    temp_video_path = video_frames_dir / f"temp_{vid_id}.mp4"

    # Use yt-dlp to download the 360p/480p stream for fast processing
    ytdlp_cmd = [
        "yt-dlp",
        "-f", "bestvideo[height<=480][ext=mp4]/best[height<=480]/best",
        "--download-sections", f"*{start_time}-{end_time}",
        "--force-keyframes-at-cuts",
        "-o", str(temp_video_path),
        url,
    ]

    try:
        print(f"Downloading clip for {chunk['game']} ({vid_id})...")
        subprocess.run(ytdlp_cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=120)

        # Extract frames using OpenCV
        if temp_video_path.exists():
            cap = cv2.VideoCapture(str(temp_video_path))
            f_idx = 0
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                abs_idx = start_frame + f_idx
                frame_path = frame_dest_dir / f"frame_{abs_idx:06d}.jpg"
                cv2.imwrite(str(frame_path), frame, [cv2.IMWRITE_JPEG_QUALITY, 85])
                f_idx += 1
            cap.release()
            temp_video_path.unlink(missing_ok=True)
            print(f"Extracted {f_idx} frames for {chunk['game']} ({vid_id})")
            downloaded_count += 1
    except Exception as err:
        print(f"Failed to download {vid_id}: {err}")
        temp_video_path.unlink(missing_ok=True)

print(f"Ready: Downloaded and extracted {downloaded_count} platformer gameplay video clips!")


In [ ]:
from src.data.nitrogen_datamodule import NitroGenDataModule

# DataModule hyperparameters (NVIDIA NitroGen Streaming Dataset)
datamodule = NitroGenDataModule(
    # Storage and streaming source
    video_dir=str(video_frames_dir),               # Local directory containing extracted real gameplay video frames
    repo_id="nvidia/NitroGen",                     # Hugging Face Hub dataset repository for actions & metadata
    shards=None,                                   # Optional list of integer shard IDs to stream (None = all shards)
    max_shards=None,                               # Maximum number of shards to process (None = unlimited)
    max_chunks_per_shard=None,                     # Maximum 20-second chunks to read per shard (None = unlimited)
    # Batching and worker resources
    batch_size=32,                                 # Per-GPU batch size (effective batch size = 64 on 2x T4)
    num_workers=2,                                 # DataLoader worker processes per GPU (optimal for Kaggle 4-vCPU)
    pin_memory=torch.cuda.is_available(),          # Pin memory tensors into page-locked CUDA RAM for faster DMA transfers
    # Sample limits and evaluation budgets
    max_samples=50000,                             # Total training sample budget per epoch
    val_samples=1000,                              # Validation sample budget per evaluation cycle
    test_samples=1000,                             # Test sample budget for post-training evaluation
    val_ratio=0.1,                                 # Fraction of video chunks reserved for validation when streaming (10%)
    # Sequence unrolling and temporal dynamics
    steps_per_sample=16,                           # Number of temporal steps per chunk window for RWKV-7 mixing
    single_step=True,                              # Unroll 16-step windows into 16 individual single-frame forward passes
    # Shuffling and random seed
    shuffle=True,                                  # Maintain continuous intra-episode frame order while shuffling episodes
    shuffle_buffer_size=1000,                      # Reservoir shuffle buffer size (number of episode windows)
    seed=3407,                                     # Random seed for deterministic sample generation and splitting
    # Spatial resolution
    image_size=(224, 224),                         # Input image spatial resolution (height, width)
)

datamodule.setup("fit")


In [ ]:
from src.models.components.convnext_rwkv7 import ConvNeXtRWKV7Gamepad
from src.models.convnext_rwkv7_module import ConvNeXtRWKV7GamepadLitModule

# Model architecture hyperparameters (ConvNeXt-RWKV7 Gamepad)
net = ConvNeXtRWKV7Gamepad(
    # Input and spatial preprocessing
    in_chans=3,                                    # Number of input image channels (3 for RGB gameplay frames)
    pool_intermediate_features=32,                 # Intermediate channels inside AdaptiveLearnedPool2d downsampler
    # ConvNeXt vision backbone
    convnext_size="tiny",                          # ConvNeXt scale ('tiny', 'small', 'base', 'large')
    convnext_dims=None,                            # Custom stage channel widths (None uses convnext_size defaults: [96, 192, 384, 768])
    convnext_depths=None,                          # Custom stage block depths (None uses convnext_size defaults: [3, 3, 9, 3])
    convnext_drop_path_rate=0.0,                   # Stochastic depth / DropPath rate for ConvNeXt residual branches
    convnext_layer_scale_init_value=1e-6,          # Initial multiplier for LayerScale in ConvNeXt blocks
    pretrained_dinov3=True,                        # Load self-supervised DINOv3 pre-trained visual representations
    dinov3_repo_id="facebook/dinov3-convnext-tiny-pretrain-lvd1689m", # Hugging Face Hub repository ID for DINOv3 weights
    bypass_stem=False,                             # If True, pooler feeds stage-0 directly (56x56) bypassing ConvNeXt stem
    freeze_convnext=True,                          # Freeze ConvNeXt weights while preserving autograd flow to AdaptiveLearnedPool2d
    # Spatial feature aggregation
    gap_kernel_size=3,                             # Kernel size for spatial attention 2D convolution in LearnedWeightedGAP
    gap_concat=True,                               # Whether LearnedWeightedGAP concatenates uniform GAP + weighted GAP features
    # Temporal and recurrent dynamics
    causal_conv_kernel_size=3,                     # 1D causal convolution kernel size along temporal sequence dimension
    rwkv_dim=256,                                  # Hidden representation channel width across RWKV-7 recurrent blocks
    rwkv_head_size=64,                             # Attention head dimension for RWKV-7 linear attention
    rwkv_layers=4,                                 # Number of stacked RWKV-7 (Goose) linear attention recurrent blocks
    rwkv_dim_ffn=None,                             # RWKV-7 Feed-Forward Network hidden dimension (None = 4 * rwkv_dim = 1024)
    # Gamepad prediction head
    head_hidden_dim=256,                           # Hidden projection layer width inside GamepadHead
    num_buttons=17,                                # Number of discrete gamepad button logits (BCE loss)
    num_joysticks=2,                               # Number of dual-axis analog joysticks (2 joysticks * 2 axes = 4 MSE values in [-1, 1])
)

# Optimizer and learning rate scheduler hyperparameters
max_epochs = 15

# AdamW optimizer factory (trains pooler, spatial GAP, causal conv, RWKV-7 blocks, and gamepad head)
optimizer_factory = partial(
    torch.optim.AdamW,
    lr=1e-3,                                       # Base learning rate for trainable model components
    weight_decay=0.01,                             # Decoupled L2 weight decay regularization
    betas=(0.9, 0.999),                            # Adam first and second momentum coefficient estimates
    eps=1e-8,                                      # Epsilon term for numerical stability in optimizer denominator
)

# Cosine annealing learning rate scheduler factory
scheduler_factory = partial(
    torch.optim.lr_scheduler.CosineAnnealingLR,
    T_max=max_epochs,                              # Maximum number of epochs for full cosine decay cycle
    eta_min=1e-6,                                  # Minimum learning rate floor at the end of the cosine schedule
)

# PyTorch Lightning module wrapper
model = ConvNeXtRWKV7GamepadLitModule(
    net=net,                                       # Instantiated ConvNeXtRWKV7Gamepad backbone network
    optimizer=optimizer_factory,                   # Partial optimizer callable
    scheduler=scheduler_factory,                   # Partial learning rate scheduler callable
    joystick_loss_weight=1.0,                      # Scalar loss multiplier weighting analog joystick MSE vs BCE button loss
    convnext_lr=None,                              # Differential LR for ConvNeXt when unfrozen (e.g. 1e-5; None = base lr)
    compile=False,                                 # Whether to JIT-compile model backbone with torch.compile
)


In [ ]:
# Hardware strategy and precision configuration
strategy = "ddp_notebook" if num_gpus > 1 else "auto"  # Distributed training strategy ('ddp_notebook' prevents fork crashes on Kaggle)
devices = num_gpus if num_gpus > 0 else "auto"         # Number of GPU devices to allocate
accelerator = "gpu" if num_gpus > 0 else "cpu"         # Hardware accelerator ('gpu', 'cpu', or 'auto')
precision = "16-mixed" if num_gpus > 0 else "32-true"  # Precision mode ('16-mixed' for FP16 Tensor Cores, '32-true' for FP32)

# Logging configuration
logger = CSVLogger(
    save_dir="logs",                                   # Base directory for logging metrics and CSV output
    name="nitrogen_platformer_rwkv7",                  # Experiment subdirectory name
)

# Training callbacks
callbacks = [
    ModelCheckpoint(
        dirpath="checkpoints/nitrogen_platformer",     # Directory where model weight checkpoints will be stored
        filename="platformer-{epoch:02d}-{val/loss:.4f}", # Checkpoint filename pattern with epoch and metric tags
        monitor="val/loss",                            # Metric key to track for best checkpoint selection
        mode="min",                                    # Monitored optimization mode ('min' for loss, 'max' for accuracy)
        save_top_k=2,                                  # Number of best checkpoints to retain on disk
        save_last=True,                                # Always keep the last checkpoint (last.ckpt) for resume support
    ),
    EarlyStopping(
        monitor="val/loss",                            # Metric key to track for early stopping
        patience=4,                                    # Number of validation checks with no improvement before stopping
        min_delta=1e-4,                                # Minimum change in monitored metric to qualify as improvement
        mode="min",                                    # Optimization direction for monitored metric
    ),
    RichProgressBar(),                                 # Clean progress bar rendering with ETA and metrics
]

# PyTorch Lightning trainer
trainer = L.Trainer(
    # Hardware and distributed
    accelerator=accelerator,                           # 'gpu' or 'cpu'
    devices=devices,                                   # Device count
    strategy=strategy,                                 # 'ddp_notebook' for multi-GPU Kaggle
    precision=precision,                               # Mixed precision training mode
    # Epochs and step budgets
    max_epochs=max_epochs,                             # Maximum total training epochs
    max_steps=-1,                                      # Total step budget limit (-1 runs for full max_epochs)
    # Optimization and regularization
    gradient_clip_val=1.0,                             # Maximum gradient norm for clipping (stabilizes recurrent RWKV-7 training)
    accumulate_grad_batches=1,                         # Gradient accumulation step count (simulates larger batch sizes)
    # Logging and validation frequency
    callbacks=callbacks,                               # List of instantiated Lightning callbacks
    logger=logger,                                     # Experiment metric logger
    log_every_n_steps=25,                              # Step interval for training metric logging
    val_check_interval=1.0,                            # Validation loop frequency (1.0 = once per epoch)
    # Debugging and reproducibility controls
    fast_dev_run=False,                                # Set to True or 1 to run 1 batch sanity check through train/val/test
    deterministic=False,                               # Set to True to enforce deterministic PyTorch operations
)


In [ ]:
# Train ConvNeXt-RWKV7 on the filtered NitroGen 2D platformer gameplay data
trainer.fit(model=model, datamodule=datamodule)

# Evaluate best checkpoint
trainer.test(model=model, datamodule=datamodule, ckpt_path="best")


In [ ]:
# Real-Time Online Recurrent Streaming Inference Demo (Frame-by-Frame)
model.eval()
device = next(model.parameters()).device

# Initialize recurrent state tuple
state = model.net.init_streaming_state(batch_size=1, device=device)

# Load a test sample
datamodule.setup("test")
test_frame, target_actions = next(iter(datamodule.test_dataloader()))
input_frame = test_frame[0:1].to(device)  # Shape: (1, 3, 224, 224)

# Execute O(1) recurrent step
with torch.no_grad():
    (full_gamepad, buttons_logits, joysticks), state = model.net.step(input_frame, state)

btn_probs = buttons_logits.sigmoid().squeeze(0).cpu().tolist()
joy_axes = joysticks.squeeze(0).cpu().tolist()

print(f"Predicted Full Gamepad Vector (21-D): {full_gamepad.shape}")
print("Predicted Button Probabilities (17-D):", [round(p, 3) for p in btn_probs])
print(f"Predicted Left Stick (X, Y): ({joy_axes[0]:.3f}, {joy_axes[1]:.3f})")
print(f"Predicted Right Stick (X, Y): ({joy_axes[2]:.3f}, {joy_axes[3]:.3f})")
